# 02 - Preprocessing Pipeline
Notebook ini menyiapkan sequence tensor dari data mentah dan menyimpan artefak untuk training.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data import discover_dataset_files, prepare_password_sequence_bundle

DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
dataset_files = discover_dataset_files(DATA_RAW_DIR)
if not dataset_files:
    raise FileNotFoundError('No dataset in data/raw. Please place DSL-StrongPasswordData.csv first.')

target_file = dataset_files[0]
# Load the tabular file into a DataFrame before preprocessing
from data import load_tabular_file, normalize_column_names, prepare_password_sequence_bundle
df = load_tabular_file(target_file)
df = normalize_column_names(df)
df = df.sort_values(['subject', 'sessionindex', 'rep']).reset_index(drop=True)
bundle = prepare_password_sequence_bundle(df, label_column='subject', session_column='sessionindex', rep_column='rep', include_terminal_key=True, normalize=False)

features = np.asarray(bundle.features)
labels = np.asarray(bundle.labels)
sequence_ids = np.asarray(bundle.sequence_ids, dtype=object)

artifact_path = DATA_PROCESSED_DIR / 'sequence_bundle.npz'
np.savez_compressed(artifact_path, features=features, labels=labels, sequence_ids=sequence_ids)

meta = {
    'source_file': str(target_file),
    'features_shape': list(features.shape),
    'labels_len': int(len(labels)),
    'sequence_length': 11,
}
meta_path = OUTPUT_REPORTS_DIR / 'preprocessing_summary.json'
meta_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')

print(f'Features shape: {features.shape}')
print(f'Labels size: {len(labels)}')
print(f'Saved artifact: {artifact_path}')
print(f'Saved summary: {meta_path}')

Features shape: (14242, 11, 3)
Labels size: 14242
Saved artifact: C:\Users\anang\Downloads\Projek Keamanan Informasi\data\processed\sequence_bundle.npz
Saved summary: C:\Users\anang\Downloads\Projek Keamanan Informasi\outputs\reports\preprocessing_summary.json
